# Twitter (microblog)

> **Time-box:** 45–60 minutes. The follow graph and the timeline are where the depth lives — don't spend it on user CRUD.

## Core requirements

1. Users have a unique handle and a display name.
2. A user can post a short tweet (≤ 280 chars).
3. A user can follow / unfollow another user. Following is one-directional.
4. A user can fetch their **home timeline**: tweets from the people they follow, newest first, paginated.
5. A user can fetch any user's **profile timeline**: that user's own tweets.
6. A user can like / unlike a tweet. A tweet shows its like count.

## Stretch goals

- Retweets (a tweet with a reference to an original).
- Replies (threading).
- Mentions of `@handle` in the body — what does the schema need?
- Fan-out: at write-time (push to followers' inboxes) vs read-time (query at read).

## Things the interviewer will probe

- **Pagination:** offset vs cursor (`created_at + id`). What happens with new inserts mid-scroll?
- **Hot follows:** one user has 100M followers. Fan-out on write becomes catastrophic. How do you handle celebrity accounts?
- **Like counts:** denormalized counter on `tweets` vs live `COUNT(*)`? Trade-offs?
- **Composite keys:** `follows` is naturally `(follower_id, followee_id)`. Do you give it a synthetic `id` anyway? Why or why not?
- **Resource naming:** `POST /users/{handle}/follow` vs `POST /follows`? Defend a choice.

---
## Setup

In [1]:
import json
import sqlite3

import pandas as pd
from fastapi import FastAPI
from fastapi.testclient import TestClient
from IPython.display import display
from pydantic import BaseModel

## Schema

Edit the SQL and re-run this cell to get a fresh in-memory database.

In [2]:
SCHEMA = """
CREATE TABLE users (
    id   INTEGER PRIMARY KEY,
    name TEXT NOT NULL
);

CREATE TABLE follows (
    follower_id INTEGER,
    followee_id INTEGER,
    PRIMARY KEY (follower_id, followee_id)
);

CREATE TABLE posts (
    id         INTEGER PRIMARY KEY,
    user_id    INTEGER NOT NULL REFERENCES users(id) ON DELETE CASCADE,
    content    TEXT    NOT NULL,
    created_at TEXT    NOT NULL DEFAULT CURRENT_TIMESTAMP
);
"""

conn = sqlite3.connect(":memory:", check_same_thread=False)
conn.row_factory = sqlite3.Row
conn.execute("PRAGMA foreign_keys = ON")
conn.executescript(SCHEMA)

print("Tables:", [r[0] for r in conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
).fetchall()])

Tables: ['follows', 'posts', 'users']


## API

> After editing any cell below, re-run from **App** down through **Client**.

In [3]:
# ── App + models ──────────────────────────────────────────────────────────────
app = FastAPI(title="Twitter")


class CreateUser(BaseModel):
    name: str


class Follow(BaseModel):
    follower_id: int
    followee_id: int


class CreatePost(BaseModel):
    user_id: int
    content: str

In [4]:
# ── Users ─────────────────────────────────────────────────────────────────────
@app.get("/healthz")
def healthz():
    return {"status": "ok"}


@app.post("/users", status_code=201)
def create_user(payload: CreateUser):
    with conn:
        conn.execute("INSERT INTO users(name) VALUES (?)", (payload.name,))
    return {"status": "ok"}

In [5]:
# ── Follows ───────────────────────────────────────────────────────────────────
@app.post("/follows", status_code=201)
def follow(payload: Follow):
    with conn:
        conn.execute(
            "INSERT INTO follows(follower_id, followee_id) VALUES (?, ?)",
            (payload.follower_id, payload.followee_id),
        )
    return {"status": "ok"}

In [6]:
# ── Posts ─────────────────────────────────────────────────────────────────────
@app.post("/posts", status_code=201)
def create_post(payload: CreatePost):
    with conn:
        conn.execute(
            "INSERT INTO posts(user_id, content) VALUES (?, ?)",
            (payload.user_id, payload.content),
        )
    return {"status": "ok"}

In [7]:
# ── Add more endpoints here ───────────────────────────────────────────────────

In [8]:
# ── Client ────────────────────────────────────────────────────────────────────
client = TestClient(app, raise_server_exceptions=True)
print(client.get("/healthz").json())

{'status': 'ok'}


## Helpers

In [9]:
def call(method: str, path: str, **kwargs):
    r = getattr(client, method)(path, **kwargs)
    body = r.json() if r.content else None
    print(f"{method.upper():6s} {path}  →  {r.status_code}")
    if body is not None:
        print(json.dumps(body, indent=2))
    return r


def df(table: str) -> pd.DataFrame:
    return pd.read_sql(f"SELECT * FROM {table}", conn)


def query(sql: str, *params) -> pd.DataFrame:
    return pd.read_sql(sql, conn, params=list(params) if params else None)


def show_all():
    names = [r[0] for r in conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
    ).fetchall()]
    for name in names:
        count = conn.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]
        print(f"\n── {name} ({count} rows) ──")
        display(pd.read_sql(f"SELECT * FROM {name}", conn))

## Demo

In [10]:
call("post", "/users", json={"name": "Alice"})
call("post", "/users", json={"name": "Bob"})
call("post", "/users", json={"name": "Carol"})
df("users")

POST   /users  →  201
{
  "status": "ok"
}
POST   /users  →  201
{
  "status": "ok"
}
POST   /users  →  201
{
  "status": "ok"
}


,id,name
0,1,Alice
1,2,Bob
2,3,Carol


In [11]:
call("post", "/follows", json={"follower_id": 1, "followee_id": 2})  # Alice → Bob
call("post", "/follows", json={"follower_id": 1, "followee_id": 3})  # Alice → Carol
call("post", "/follows", json={"follower_id": 2, "followee_id": 1})  # Bob → Alice
df("follows")

POST   /follows  →  201
{
  "status": "ok"
}
POST   /follows  →  201
{
  "status": "ok"
}
POST   /follows  →  201
{
  "status": "ok"
}


,follower_id,followee_id
0,1,2
1,1,3
2,2,1


In [12]:
call("post", "/posts", json={"user_id": 2, "content": "Hello from Bob!"})
call("post", "/posts", json={"user_id": 3, "content": "Carol here. Hi everyone!"})
call("post", "/posts", json={"user_id": 2, "content": "Bob's second post."})
call("post", "/posts", json={"user_id": 1, "content": "Alice's first post."})
df("posts")

POST   /posts  →  201
{
  "status": "ok"
}
POST   /posts  →  201
{
  "status": "ok"
}
POST   /posts  →  201
{
  "status": "ok"
}
POST   /posts  →  201
{
  "status": "ok"
}


,id,user_id,content,created_at
0,1,2,Hello from Bob!,2026-05-24 09:22:02
1,2,3,Carol here. Hi everyone!,2026-05-24 09:22:02
2,3,2,Bob's second post.,2026-05-24 09:22:02
3,4,1,Alice's first post.,2026-05-24 09:22:02


In [13]:
# Add demo calls for new endpoints here

## All tables

In [14]:
show_all()


── follows (3 rows) ──


,follower_id,followee_id
0,1,2
1,1,3
2,2,1



── posts (4 rows) ──


,id,user_id,content,created_at
0,1,2,Hello from Bob!,2026-05-24 09:22:02
1,2,3,Carol here. Hi everyone!,2026-05-24 09:22:02
2,3,2,Bob's second post.,2026-05-24 09:22:02
3,4,1,Alice's first post.,2026-05-24 09:22:02



── users (3 rows) ──


,id,name
0,1,Alice
1,2,Bob
2,3,Carol
